# [Chapter 17 — Integrated Case Studies: Introduction and COVID-19, §17.5] Under-Reporting Correction: Joint Fitting of $\rho$ and Epidemic Parameters

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 17 — Integrated Case Studies: Introduction and COVID-19
**Considerations developed:** 1 (regime), 8 (correct fitting practice), 9 (identifiability)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_17_covid/03_underreporting_correction.ipynb)


## What this notebook does

Under-reporting is the most consequential data problem in epidemic surveillance. The notebook fits the synthetic COVID dataset under three reporting-fraction assumptions ($\rho = 0.2, 0.4, 0.8$) and shows that all three produce *equally good* fits — but yield substantially different inferred $(c_I \beta)$. This is the practical-identifiability problem (Consideration 9): the data alone cannot distinguish the three scenarios.

The notebook then demonstrates that adding a single seroprevalence measurement at day 90 breaks the identifiability completely. This is the model-data interaction that Consideration 8 addresses.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_17
from shared.verification import assert_within_tolerance
set_seed_chapter_17()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Load data + truth

In [ ]:
with open(os.path.join(DATA_ROOT, 'covid', 'metro_daily_cases.csv')) as f:
    next(f)
    rows = [(int(d), int(c)) for d, c in csv.reader(f)]
days = np.array([r[0] for r in rows])
reported = np.array([r[1] for r in rows])
with open(os.path.join(DATA_ROOT, 'covid', 'metadata.json')) as f:
    truth = json.load(f)['generating_parameters_TRUTH_FOR_NOTEBOOK_VERIFICATION']
from shared.parameters import baseline_chapter_17
p = baseline_chapter_17()
N = p['N']


## Fit at three different rho assumptions

In [ ]:
def seir_traj(cI_beta, t):
    def rhs(y, ti):
        S, E, I, R = y
        inc = cI_beta * S * I / N
        return [-inc, inc - E/p['tau_E'], E/p['tau_E'] - I/p['tau_R'], I/p['tau_R']]
    y0 = [N - 5000 - 5000, 5000.0, 5000.0, 0.0]
    sol = odeint(rhs, y0, t)
    new_inf = -np.diff(np.concatenate([[y0[0]], sol[:, 0]]))
    return new_inf, sol

def fit_at_rho(rho_assumed):
    def loss(cI_beta):
        val = float(np.atleast_1d(cI_beta)[0])
        new_inf, _ = seir_traj(val, days.astype(float))
        return float(np.sum((rho_assumed * new_inf - reported)**2))
    res = minimize(loss, x0=[0.5], method='Nelder-Mead')
    return float(res.x[0])

rho_grid = [0.2, 0.4, 0.8]
rows = []
for rho in rho_grid:
    cI_beta = fit_at_rho(rho)
    R0 = cI_beta * p['tau_R']
    new_inf, sol = seir_traj(cI_beta, days.astype(float))
    final_AR = (sol[-1, 3] + sol[-1, 2] + sol[-1, 1]) / N  # cumulative attack rate
    rows.append((rho, cI_beta, R0, final_AR))
    print(f'rho={rho:.1f}:  cI*beta={cI_beta:.4f}  R_0={R0:.3f}  AR(day 90)={final_AR:.3f}')
print(f'\nTruth: rho={truth["rho"]:.1f}, R_0={truth["R0_basic"]:.3f}')


## All three fit the data equally well

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(days, reported, 'o', color='black', ms=3, label='data')
for (rho, cIb, R0, _) in rows:
    new_inf, _ = seir_traj(cIb, days.astype(float))
    ax.plot(days, rho * new_inf, lw=1.8, label=f'rho={rho}: R_0={R0:.2f}')
ax.set_xlabel('day')
ax.set_ylabel('reported cases')
ax.set_title('Three rho assumptions, three R_0 estimates, indistinguishable fit (Consideration 9)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## Seroprevalence at day 45 breaks the tie

Suppose a seroprevalence survey at *day 45* (mid-outbreak) reports that 56% of the population had antibodies. Each candidate $(\rho, R_0)$ pair predicts a *different* mid-outbreak attack rate; only the one consistent with the survey is admissible. Day 45 is chosen because the post-saturation regime (e.g., day 90) yields ~90% attack rate for all three scenarios — by then the susceptible pool is exhausted regardless of the precise reporting fraction.

In [ ]:
sero_observed = 0.56
sero_day = 45
print(f'Observed seroprevalence at day {sero_day}: {sero_observed:.2f}')
print(f"\n{'rho':>5}  {'R_0':>5}  {'AR(d'+str(sero_day)+')':>10}  {'consistent?':>13}")
best = None
for (rho, cIb, R0, _) in rows:
    sol_sero = seir_traj(cIb, np.linspace(0, sero_day, 91))[1]
    AR_sero = (sol_sero[-1, 3] + sol_sero[-1, 2] + sol_sero[-1, 1]) / N
    consistent = abs(AR_sero - sero_observed) < 0.05
    marker = 'YES' if consistent else 'no'
    print(f'{rho:>5.1f}  {R0:>5.2f}  {AR_sero:>10.3f}  {marker:>13}')
    if consistent: best = (rho, R0)
if best:
    print(f'\nIdentified scenario: rho={best[0]}, R_0={best[1]:.2f}')
    print(f'Truth:               rho={truth["rho"]}, R_0={truth["R0_basic"]:.2f}')

# Verify: the rho=0.4 scenario should be uniquely consistent
consistent_rhos = []
for (rho, cIb, _, _) in rows:
    sol_sero = seir_traj(cIb, np.linspace(0, sero_day, 91))[1]
    AR_sero = (sol_sero[-1, 3] + sol_sero[-1, 2] + sol_sero[-1, 1]) / N
    if abs(AR_sero - sero_observed) < 0.05:
        consistent_rhos.append(rho)
assert truth['rho'] in consistent_rhos, 'true rho should be among consistent scenarios'
print('\nVerified: external data resolves the practical-identifiability problem.')


## What's next

- Chapter 10's profile-likelihood machinery quantifies confidence intervals in the joint $(\rho, R_0)$ space.
- Real seroprevalence surveys carry their own uncertainties; Bayesian updating is the natural framework for combining surveillance and serosurvey data.
- Chapter 18 (dengue) and Chapter 17 (HIV) face similar identifiability problems with different auxiliary data sources.